# Example notebook for customizing circuit populations

This notebook demonstrates how to create a customized circuit by:
1. Downloading a parent circuit from entitycore
2. Replacing node/edge population files, node sets, or the circuit config
3. Validating the customized circuit
4. Uploading the result as a new circuit entity with a derivation link

## Setup

In [ ]:
from pathlib import Path

from entitysdk import Client, ProjectContext, models
from obi_auth import get_token

from obi_one.utils.circuit_customization.populations import create_modified_circuit
from obi_one.utils.circuit_customization.upload import upload_customized_circuit
from obi_one.utils.circuit_registration.links import CustomizationType

In [ ]:
# Disable info logging (optional)
import logging
logging.getLogger("httpx").setLevel(logging.WARNING)

In [ ]:
# "Shared results + tests"
token = get_token(environment="staging")
project_context = ProjectContext(virtual_lab_id="e6030ed8-a589-4be2-80a6-f975406eb1f6", project_id="2720f785-a3a2-4472-969d-19a53891c817")
client = Client(environment="staging", project_context=project_context, token_manager=token)


## Configuration

Specify the parent circuit ID and output path for the customized circuit.

In [ ]:
# Parent circuit ID (from entitycore)
PARENT_CIRCUIT_ID = "39334a5e-90bd-4d8b-b5b8-8ac3ac7be2c1"

# Custom input file path
CUSTOM_FILE_PATH = Path("./custom_files")

# Output path for the new customized circuit
NEW_CIRCUIT_PATH = Path("./customized_circuit")

In [ ]:
# Print parent circuit information
parent_circuit = client.get_entity(entity_id=PARENT_CIRCUIT_ID, entity_type=models.Circuit)
print(f"{parent_circuit.name} (ID {parent_circuit.id})")
print(parent_circuit.description)
print(f"#Neurons: {parent_circuit.number_neurons}, #Synapses: {parent_circuit.number_synapses}, #Connections: {parent_circuit.number_connections}, Scale: {parent_circuit.scale}")

## Prepare customization files

Provide paths to any replacement files. Set to `None` if not replacing.

- `new_circuit_config_path`: Replacement `circuit_config.json` (e.g., if adding/removing populations)
- `new_node_sets_path`: Replacement `node_sets.json`
- `new_node_population_paths`: Dict mapping population name to replacement or additional `.h5` file
- `new_edge_population_paths`: Dict mapping population name to replacement or additional `.h5` file

In this example, we will remove the two of the edge populations, so we need to
- download the exiting circuit config file
- remove the edge populations from the config file
- use it as customization

In [ ]:
# Download existing circuit config
from obi_one.utils.circuit_customization.download import download_circuit_config

config_path = download_circuit_config(PARENT_CIRCUIT_ID, client, CUSTOM_FILE_PATH)
print(f"Circuit config downloaded to: {config_path}")


In [ ]:
# Remove the edge population from the config file
import json

with config_path.open("r") as f:
    config_dict = json.load(f)

rm_epops = ["S1nonbarrel_neurons__S1nonbarrel_neurons__chemical", "external_S1nonbarrel_neurons__S1nonbarrel_neurons__chemical"]
epop_list = [epop for epop in config_dict["networks"]["edges"] if not any(rm in epop["populations"] for rm in rm_epops)]
config_dict["networks"]["edges"] = epop_list

with config_path.open("w") as f:
    json.dump(config_dict, f, indent=2)

In [ ]:
# Add custom file paths
# TODO: Additional customizations may be added here

new_circuit_config_path = config_path  # Path to new circuit_config.json, or None

new_node_sets_path = None  # Path to new node_sets.json, or None

new_node_population_paths = None  # e.g., {"S1nonbarrel_neurons": Path("./custom_nodes.h5")}
new_edge_population_paths = None  # e.g., {"S1nonbarrel_neurons__S1nonbarrel_neurons__chemical": Path("./custom_edges.h5")}

## Create customized circuit

This will:
1. Download the parent circuit directory (writable copy)
2. Replace the specified files
3. Validate the result (loads circuit, checks morphologies and electrical models)

In [ ]:
new_circuit_path, parent_entity = create_modified_circuit(
    db_client=client,
    circuit_id=PARENT_CIRCUIT_ID,
    new_circuit_config_path=new_circuit_config_path,
    new_node_sets_path=new_node_sets_path,
    new_node_population_paths=new_node_population_paths,
    new_edge_population_paths=new_edge_population_paths,
    new_circuit_path=NEW_CIRCUIT_PATH,
)

print(f"Customized circuit created at: {new_circuit_path}")
print(f"Parent circuit: {parent_entity.name} ({parent_entity.id})")

## Upload customized circuit

Register the customized circuit as a new entity in entitycore, with a derivation link to the parent.

In [ ]:
dry_run = False  # If True, runs all checks but no actual registration; otherwise actual registration

In [ ]:
registered_circuit = upload_customized_circuit(
    client=client,
    name=f"CustomCircuit01-{parent_entity.name}",
    description="Removed internal and external edge populations. Kept projections only.",
    circuit_path=new_circuit_path,
    customized_from=parent_entity,
    customization_type=CustomizationType.population_modification,
    dry_run=dry_run,
)

if registered_circuit:
    print(f"Registered customized circuit: {registered_circuit.name} (ID {registered_circuit.id})")
    print(registered_circuit.description)
    print(f"#Neurons: {registered_circuit.number_neurons}, #Synapses: {registered_circuit.number_synapses}, #Connections: {registered_circuit.number_connections}, Scale: {registered_circuit.scale}")
else:
    print("Customized circuit not registered!")